# ブロック崩し（p5.js ゲーム）

パドルでボールを跳ね返し、上のブロックをすべて壊すゲームです。

## ルール
- ボールがブロックに当たるとブロックが消えて **10 点**
- ボールを下に落とすとライフが 1 つ減り、ライフが 0 でゲームオーバー
- ブロックを全部壊すとクリア

## 操作
- **マウスを左右に動かす**: パドルを移動
- **クリック**: ボールを発射 / ゲームオーバー・クリア後にもう一度遊ぶ

## このノートブックの使い方

- コードセルを上から順番に **Shift + Enter** で実行し、最後の `%show` セルを実行するとゲーム画面が表示されます。
- キーボードで操作するゲームは、**最初にゲーム画面をクリック** してから操作してください（クリックでキー入力が画面に届くようになります）。
- コードを書き換えたら、そのセルを実行し直してから `%show` をもう一度実行すると、新しいゲームになります。
- 動かなくなったら、メニューの **Kernel → Restart Kernel and Clear Outputs of All Cells...** で最初からやり直せます。

p5.js の基本は `p5-tutorial.ipynb` で学べます。

## 1. ゲームの状態

In [ ]:
let paddleX = 200;        // パドルの中心 x
const PADDLE_W = 80;      // パドルの幅
const PADDLE_H = 12;

let ballX = 200;          // ボールの位置と速度
let ballY = 300;
let ballVX = 3;
let ballVY = -4;
const BALL_R = 8;         // ボールの半径

let bricks = [];          // ブロックの配列
let score = 0;
let lives = 3;
let state = "ready";      // "ready"（発射待ち）, "play", "gameover", "clear"

## 2. ブロックのクラスと配置

ブロックは「左上の座標・幅・高さ・色・生きているか」を持ちます。`makeBricks()` で 5 行 × 8 列に並べます。

In [ ]:
class Brick {
  constructor(x, y, w, h, c) {
    this.x = x;
    this.y = y;
    this.w = w;
    this.h = h;
    this.c = c;
    this.alive = true;
  }

  show() {
    if (!this.alive) return;
    fill(this.c);
    stroke(255);
    rect(this.x, this.y, this.w, this.h);
  }

  // ボール（中心 bx, by, 半径 r）がこのブロックに当たっているか
  hits(bx, by, r) {
    return this.alive &&
      bx + r > this.x && bx - r < this.x + this.w &&
      by + r > this.y && by - r < this.y + this.h;
  }
}

function makeBricks() {
  bricks = [];
  const cols = 8;
  const rows = 5;
  const w = width / cols;
  const h = 20;
  const colors = ["#e74c3c", "#e67e22", "#f1c40f", "#2ecc71", "#3498db"];
  for (let row = 0; row < rows; row++) {
    for (let col = 0; col < cols; col++) {
      bricks.push(new Brick(col * w, 40 + row * h, w, h, colors[row]));
    }
  }
}

## 3. setup と draw

ボールの動きは「位置に速度を足す → 壁・パドル・ブロックとの当たり判定で速度を反転」の繰り返しです。
パドルの **どこに当たったか** で跳ね返る角度を変えると、狙って打てるようになります。

In [ ]:
function setup() {
  createCanvas(400, 400);
  textFont("sans-serif");
  makeBricks();
}

function draw() {
  background(20, 20, 40);

  // パドル
  paddleX = constrain(mouseX, PADDLE_W / 2, width - PADDLE_W / 2);
  noStroke();
  fill(230);
  rect(paddleX - PADDLE_W / 2, height - 30, PADDLE_W, PADDLE_H, 4);

  // ブロック
  for (const b of bricks) b.show();

  if (state === "ready") {
    // 発射待ち: ボールはパドルの上に乗せておく
    ballX = paddleX;
    ballY = height - 30 - BALL_R;
  } else if (state === "play") {
    moveBall();
  }

  // ボール
  noStroke();
  fill(255, 220, 0);
  circle(ballX, ballY, BALL_R * 2);

  drawHUD();
}

function moveBall() {
  ballX += ballVX;
  ballY += ballVY;

  // 左右の壁と天井
  if (ballX < BALL_R || ballX > width - BALL_R) ballVX *= -1;
  if (ballY < BALL_R) ballVY *= -1;

  // パドル（ボールが下向きに動いているときだけ判定）
  if (ballVY > 0 &&
      ballY + BALL_R >= height - 30 && ballY + BALL_R <= height - 30 + PADDLE_H &&
      abs(ballX - paddleX) < PADDLE_W / 2 + BALL_R) {
    ballVY = -abs(ballVY);
    // パドルの中心からのずれで横方向の速度を決める（端に当たるほど鋭角に）
    ballVX = map(ballX - paddleX, -PADDLE_W / 2, PADDLE_W / 2, -5, 5);
  }

  // ブロック
  for (const b of bricks) {
    if (b.hits(ballX, ballY, BALL_R)) {
      b.alive = false;
      ballVY *= -1;
      score += 10;
      break;                     // 1 フレームに 1 個だけ壊す
    }
  }

  // 全部壊したらクリア
  if (bricks.every((b) => !b.alive)) {
    state = "clear";
  }

  // 下に落ちた
  if (ballY > height + BALL_R) {
    lives--;
    if (lives <= 0) {
      state = "gameover";
    } else {
      state = "ready";
      ballVX = 3;
      ballVY = -4;
    }
  }
}

function drawHUD() {
  fill(255);
  textSize(16);
  textAlign(LEFT, TOP);
  text("スコア: " + score, 10, 10);
  textAlign(RIGHT, TOP);
  text("ライフ: " + lives, width - 10, 10);

  textAlign(CENTER, CENTER);
  if (state === "ready") {
    textSize(16);
    text("クリックで発射", width / 2, height / 2 + 40);
  } else if (state === "gameover" || state === "clear") {
    fill(0, 170);
    rect(0, 0, width, height);
    fill(255);
    textSize(36);
    text(state === "clear" ? "クリア！" : "ゲームオーバー", width / 2, height / 2 - 20);
    textSize(16);
    text("スコア: " + score + "　クリックでもう一度", width / 2, height / 2 + 25);
  }
}

## 4. 入力とリセット

In [ ]:
function mousePressed() {
  if (state === "ready") {
    state = "play";
  } else if (state === "gameover" || state === "clear") {
    resetGame();
  }
}

function resetGame() {
  makeBricks();
  score = 0;
  lives = 3;
  ballVX = 3;
  ballVY = -4;
  state = "ready";
}

In [ ]:
%show 100% 410px

## 改造のヒント

- ブロックの `rows` / `cols` や色を変えて、ステージを作ってみましょう
- 「2 回当てないと壊れないブロック」を追加してみましょう（`Brick` に耐久値を持たせる）
- ブロックを壊すたびにボールを少し速くしてみましょう
- パドルを `keyPressed()` で左右キーでも動かせるようにしてみましょう